# Architekturvergleich und Optuna-Studien

Dieses Notebook liest eine laufende oder abgeschlossene Studie. Es startet kein Training.
Alle Kandidaten verwenden dieselben Rewards, Validierungspartien und Entscheidungslimits.
Das Ranking verwendet die **letzte Validierung beim festen Budget**. `best.pt` aus dem normalen PPO-Training folgt einem anderen Auswahlkriterium; für den Studienvergleich gilt `latest.pt`.

Zunächst installieren: `python -m pip install -r ai/requirements-tuning.txt`.
Den Studienordner in der nächsten Zelle auswählen.

In [ ]:
from pathlib import Path
import csv
import json
import math
import sys
from html import escape

import matplotlib.pyplot as plt
from IPython.display import HTML, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ai/train.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from ai.tuning.optimize import open_study

STUDY_DIR = ROOT / 'runs/optuna/muc-architecture'
SMOOTHING = 10

if (STUDY_DIR / 'study.sqlite3').exists():
    study = open_study(STUDY_DIR)
    config = study.user_attrs['experiment']
    trials = study.trials
    print(f'{study.study_name}: {len(trials)} Versuche; Modus {config["mode"]}')
else:
    study, config, trials = None, None, []
    print(f'Noch keine Studie unter {STUDY_DIR}. Zuerst ai.tuning.optimize starten.')


def table(headers, rows):
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(value))}</td>' for value in row) + '</tr>' for row in rows)
    head = ''.join(f'<th>{escape(str(value))}</th>' for value in headers)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))


def run_path(trial):
    # Local artifacts can also be moved together with the study directory.
    return STUDY_DIR / f'trial-{trial.number:04d}'


def read_csv(path):
    if not path.exists():
        return []
    with path.open(encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))


def read_progress(trial):
    path = run_path(trial) / 'progress.jsonl'
    records = []
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                pass  # A running process may currently be writing the final line.
    return records


def rolling(values, width=SMOOTHING):
    return [sum(values[max(0, i-width+1):i+1]) / len(values[max(0, i-width+1):i+1]) for i in range(len(values))]

## Vergleich beim gleichen Budget

Der Score ist `Anzahl Siege + 0,49 × mittlerer Fortschritt`. Fortschritt ist das Produkt aus dem Anteil zurückgelegter Entfernung und dem Anteil erledigter Vorbereitungen (jeweils auf 0–1 begrenzt). Sicherheitsabbrüche bekommen keinen Fortschrittsbonus. Ein zusätzlicher Sieg hat damit immer Vorrang. Der Score ist eine Suchhilfe und kein Erfolgsnachweis.

Vergleiche vollständige Versuche miteinander. Abgebrochene Versuche haben weniger Trainingsschritte und werden getrennt markiert. Auch mit festem Seed unterscheiden sich Modellinitialisierung und besuchte Zustände zwischen Architekturen.

In [ ]:
rows = []
for trial in trials:
    validation = trial.user_attrs.get('validation', {})
    parameters = trial.user_attrs.get('train_parameters', {})
    elapsed = trial.user_attrs.get('elapsed_seconds', 0)
    rows.append([trial.number, trial.state.name, trial.params.get('architecture', '–'),
                 trial.user_attrs.get('parameters', '–'), f'{trial.value:.4f}' if trial.value is not None else '–',
                 f'{validation.get("wins", "–")}/{validation.get("games", "–")}',
                 f'{validation.get("progress", 0):.3f}', validation.get('total_steps', '–'),
                 f'{parameters.get("lr", 0):.2g}', f'{parameters.get("entropy_coef", 0):.3g}', f'{elapsed / 60:.1f}'])
table(['Trial', 'Status', 'Netz', 'Parameter', 'Score', 'Siege', 'Fortschritt', 'Schritte', 'Lernrate', 'Entropiegewicht', 'Minuten'], rows)

## Lernkurven

Die Leistungsplots zeigen Argmax-Validierung auf festen Würfel-Seeds. Die Value-Loss ist dagegen der Fehler der Wertvorhersage auf den jeweils neu gesammelten Trainingsdaten: Sie misst keine Spielstärke und muss nicht stetig fallen. Die gestrichelte Linie markiert die maximale Aufgabenanzahl der Strecke.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 15))
colors = plt.get_cmap('tab10')
for index, trial in enumerate(trials):
    records = read_progress(trial)
    if not records:
        continue
    color = colors(index % 10)
    style = '--' if trial.state.name == 'PRUNED' else '-'
    label = f'{trial.number}: {trial.params.get("architecture", "?")} ({trial.state.name})'
    x = [r['total_steps'] for r in records]
    for ax, key, factor in ((axes[0, 0], 'win_rate', 100), (axes[0, 1], 'mean_distance_remaining', 1),
                            (axes[1, 1], 'progress', 1)):
        ax.plot(x, [factor * r[key] for r in records], style, color=color, label=label)
    axes[1, 0].plot(x, [config['initial_tasks'] - r['mean_tasks_undone'] for r in records], style, color=color, label=label)
    metrics = read_csv(run_path(trial) / 'metrics.csv')
    # A write may be in flight; only use complete numeric update rows.
    points = []
    for row in metrics:
        try:
            value, steps = float(row['value_loss']), int(row['total_steps'])
            if math.isfinite(value) and value >= 0:
                points.append((steps, max(value, 1e-12)))
        except (ValueError, TypeError, KeyError):
            continue
    if points:
        mx, my = zip(*points)
        axes[2, 0].plot(mx, my, color=color, alpha=0.12)
        axes[2, 0].plot(mx, rolling(my), color=color, label=label)
    speed = trial.user_attrs.get('steps_per_second')
    if speed:
        axes[2, 1].bar(str(trial.number), speed, color=color)

for ax, title in zip(axes.flat, ['Gewinnrate (Validierung)', 'Mittlere Restentfernung',
                                'Erledigte Vorbereitungen', 'Gemeinsamer Navigations- und Vorbereitungsfortschritt',
                                'Value-Loss (Trainingsdaten)', 'Entscheidungen pro Sekunde inklusive Validierung']):
    ax.set_title(title)
    ax.set_xlabel('Spielentscheidungen' if ax is not axes[2, 1] else 'Trial')
    ax.grid(alpha=0.2)
    if ax.lines:
        ax.legend(fontsize=8)
axes[0, 0].set_ylabel('Siege (%)')
axes[0, 0].set_ylim(-1, 101)
axes[0, 1].set_ylabel('Entfernungsfelder')
axes[1, 0].set_ylabel('Erledigte Aufgaben')
if config:
    maximum = config['initial_tasks']
    axes[1, 0].axhline(maximum, color='green', linestyle='--', label=f'Maximum: {maximum}')
    axes[1, 0].set_ylim(-0.1, maximum + 0.1)
    axes[1, 0].legend(fontsize=8)
axes[1, 1].set_ylim(-0.01, 1.01)
axes[2, 0].set_yscale('log')
axes[2, 1].set_ylabel('Entscheidungen/s')
fig.tight_layout()
plt.show()

## Kapazität, Lernrate und Entropiegewicht

Punkte mit vollständigem Budget zeigen den erreichten Score. Einzelne Punkte reichen nicht für belastbare Aussagen über die Wirkung eines Parameters. Bei der Architektursuche sind Lernrate und Entropiegewicht absichtlich konstant.

In [ ]:
completed = [t for t in trials if t.state.name == 'COMPLETE']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, key, title in zip(axes, ['parameters', 'lr', 'entropy_coef'], ['Parameteranzahl', 'Lernrate', 'Entropiegewicht']):
    for index, trial in enumerate(completed):
        x = trial.user_attrs.get(key) if key == 'parameters' else trial.user_attrs['train_parameters'][key]
        ax.scatter(x, trial.value, color=colors(index % 10))
        ax.annotate(str(trial.number), (x, trial.value), xytext=(4, 4), textcoords='offset points')
    if key != 'parameters':
        ax.set_xscale('log')
    ax.set_xlabel(title)
    ax.set_ylabel('Score bei vollständigem Budget')
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

if completed:
    best = study.best_trial
    print(f'Gewählte Konfiguration: Trial {best.number}, {best.params}')
    print(f'Checkpoint für den fairen Vergleich: {run_path(best) / "latest.pt"}')
    print(f'Bestätigung: python -m ai.tuning.retrain --study-dir {STUDY_DIR} --trial {best.number} --seeds 101 102 103')

## Bestätigung mit unabhängigen Trainings-Seeds

`ai.tuning.retrain` trainiert die gewählte Konfiguration mit neuen Trainings-Seeds und demselben vollen Budget. Für einen fairen Vergleich auch Trial 0 (Baseline) bestätigen. Diese Läufe verwenden weiterhin dieselben Validierungspartien. Erst nach der Auswahl einmalig auf frischen Test-Seeds auswerten, beispielsweise mit `ai.evaluate --seed 2000000 --games 1000`.

Die Streuung über Trainings-Seeds wird getrennt angezeigt. Die Partien verschiedener Trainingsläufe sind wegen der gleichen Validierungs-Seeds keine unabhängige zusätzliche Stichprobe.

In [ ]:
confirmation_rows = []
for path in sorted((STUDY_DIR / 'confirmation').glob('trial-*/summary.json')):
    summary = json.loads(path.read_text(encoding='utf-8'))
    for row in summary['training_seeds']:
        confirmation_rows.append([summary['trial'], row['seed'], row['wins'], row['games'],
                                  row['win_rate'] * 100, row['mean_distance_remaining'], row['mean_tasks_undone']])
table(['Trial', 'Trainings-Seed', 'Siege', 'Partien', 'Gewinnrate (%)', 'Restentfernung', 'Offene Aufgaben'], confirmation_rows)
if confirmation_rows:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for row in confirmation_rows:
        label = f'T{row[0]}/S{row[1]}'
        axes[0].scatter(label, row[4])
        axes[1].scatter(label, row[5])
    for ax in axes:
        ax.tick_params(axis='x', rotation=45)
        ax.grid(alpha=0.2)
    axes[0].set_ylabel('Gewinnrate (%)')
    axes[1].set_ylabel('Mittlere Restentfernung')
    fig.tight_layout()
    plt.show()
else:
    print('Noch keine vollständigen Bestätigungsläufe vorhanden.')